# Building and Deploying a GAN Streamlit Web App on Heroku [Part 1]

Run from a fresh Python 3.11 kernel. Dependencies install in this kernel. Application files are checksum-verified from a local clone or the companion repository. All writes use an owned temporary workspace. No API keys required. The original unavailable Keras H5 models are replaced with both official CycleGAN PyTorch generators with the user’s approval. Original code remains verbatim below. CPU inference, no training. 

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,importlib.util,socket,threading,time
from pathlib import Path
from tempfile import TemporaryDirectory
from io import BytesIO
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check', 'torch==2.10.0','torchvision==0.25.0','streamlit==1.66.1','numpy==2.3.5','pillow==12.0.0','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
import numpy as np
import torch
from PIL import Image
from IPython.display import display
torch.set_num_threads(2)
workspace=TemporaryDirectory(prefix='tdb-coding09-')
work=Path(workspace.name)
files={'series/cyclegan-streamlit-deployment/runtime.py': '5163cacdd457dfa6c4eecfd048ed05348b55da641ab204dfafb659dbbc2e7bf3', 'series/cyclegan-streamlit-deployment/app.py': '1212d15d51d2227521f2b3951504a2171275bff9d51282bec1e9ed79e0927b7d', 'series/cyclegan-streamlit-deployment/requirements.txt': '7b2ed30444da253f070e3bc16de943cb89cef0271062b0d53173f7d6e6f1a3db', 'series/cyclegan-streamlit-deployment/Procfile': '08c86cded460b7cd6f4e9292040691b06f2f44a3b4c9a4aad1de5fbf7986a791', 'series/cyclegan-streamlit-deployment/.python-version': '49a506dd32096b010d75205acf3430c9ae6c40351888129499e5a5e487126c93'}
local_candidates=[p for p in [Path.cwd(),*Path.cwd().parents] if (p/'README.md').exists()]
local_repo=next((p for p in local_candidates if (p/next(iter(files))).exists()),None)
for relative,digest in files.items():
    target=work/relative;target.parent.mkdir(parents=True,exist_ok=True)
    if local_repo is not None:
        payload=(local_repo/relative).read_bytes()
    else:
        url='https://raw.githubusercontent.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/main/'+relative
        with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    target.write_bytes(payload)
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__)
print('Pinned dependencies and checksum-verified application files ready.')
app_dir=work/"series/cyclegan-streamlit-deployment"
sys.path.insert(0,str(app_dir))
os.environ['TDB_CYCLEGAN_CACHE']=str(work/'models')
from runtime import MODELS,SOURCE_BASE,download,load_model,load_image,generate_image
cache=work/'models'
print('Official CycleGAN 9-residual-block networks, verified replacement weights and original example images.')

In [ ]:
import venv
venv.EnvBuilder(with_pip=True,symlinks=os.name!="nt").create(work/'gan_application')
env_python=work/'gan_application'/('Scripts/python.exe'if os.name=='nt'else'bin/python')
assert env_python.exists()
print('Created isolated gan_application environment.')

In [ ]:
result=subprocess.run([str(env_python),'-c','import sys;print(sys.prefix != sys.base_prefix)'],check=True,capture_output=True,text=True)
assert result.stdout.strip()=='True'
print('Explicit environment interpreter is isolated; shell activation is optional.')

In [ ]:
import streamlit
print('Streamlit:',streamlit.__version__,'| NumPy:',np.__version__,'| Pillow:',Image.__version__)

In [ ]:
from streamlit.testing.v1 import AppTest
print('Streamlit AppTest, Pillow, NumPy and PyTorch imports ready.')

In [ ]:
models={selection:load_model(selection,cache)for selection in MODELS}
for name,model in models.items():
    assert not model.training
    print(name,'| parameters:',sum(p.numel()for p in model.parameters()))

In [ ]:
sources={}
for name in ['horse.jpg','zebra.jpg','white horse.jpg','zebra 2.jpg']:
    sources[name]=download(SOURCE_BASE+urllib.parse.quote(name),cache/name)
images={name:load_image(path)for name,path in sources.items()}
for name,image in images.items():
    assert tuple(image.shape)==(1,3,256,256)
    print(name,'| shape:',tuple(image.shape),'| range:',round(float(image.min()),4),round(float(image.max()),4))

In [ ]:
predictions={}
for name,image in images.items():
    selection='Horse 2 Zebra GAN'if 'horse'in name else'Zebra 2 Horse GAN'
    result=generate_image(models[selection],image)
    predictions[name]=result
    display(Image.fromarray((result*255).round().astype('uint8')))
    print(name,'→',selection,'| output:',result.shape,'| finite:',bool(np.isfinite(result).all()))

In [ ]:
app_source=(app_dir/'app.py').read_text()
compile(app_source,'app.py','exec')
print('Streamlit title and complete application compile successfully.')

In [ ]:
at=AppTest.from_file(str(app_dir/'app.py'),default_timeout=120).run()
assert not at.exception
assert list(at.selectbox[0].options)==list(MODELS)
print('Initial Streamlit page and both model choices render without exceptions.')

In [ ]:
for selection in MODELS:
    at.selectbox[0].select(selection).run()
    at.button[0].click().run()
    assert not at.exception
    assert at.subheader[-1].value=='Generated image'
    print('Default-image button passed:',selection)
print('Original uploaded horse and zebra files also passed the shared upload preprocessing/inference path in example6.')

In [ ]:
server_socket=socket.socket();server_socket.bind(('127.0.0.1',0));port=server_socket.getsockname()[1];server_socket.close()
process=subprocess.Popen([sys.executable,'-m','streamlit','run','app.py','--server.address=127.0.0.1',f'--server.port={port}','--server.headless=true','--browser.gatherUsageStats=false'],cwd=app_dir,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
try:
    for attempt in range(100):
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{port}/_stcore/health',timeout=2)as response:
                if response.read()==b'ok':break
        except OSError:time.sleep(.1)
    else:raise AssertionError('Streamlit health check failed')
    print('Actual local Streamlit server health endpoint returned ok.')
finally:
    process.terminate();log,_=process.communicate(timeout=30)
    print('Owned Streamlit server stopped.')

In [ ]:
# Independent preprocessing uses torchvision and the original Pillow conversion.
from torchvision.transforms import Compose,Resize,ToTensor,Normalize,InterpolationMode
reference=Compose([Resize((256,256),InterpolationMode.BICUBIC),ToTensor(),Normalize((.5,.5,.5),(.5,.5,.5))])
for name,path in sources.items():
    with Image.open(path)as original:expected=reference(original.convert('RGB')).unsqueeze(0)
    torch.testing.assert_close(images[name],expected,rtol=0,atol=1.2e-7)
    uploaded=load_image(BytesIO(path.read_bytes()))
    torch.testing.assert_close(images[name],uploaded,rtol=0,atol=0)
    selection='Horse 2 Zebra GAN'if 'horse'in name else'Zebra 2 Horse GAN'
    with torch.inference_mode():raw=models[selection](expected)
    direct=((raw[0].permute(1,2,0).numpy()+1)/2).clip(0,1)
    np.testing.assert_allclose(predictions[name],direct,rtol=1e-4,atol=2e-5)
    assert predictions[name].std()>.01
# Exercise actual encoded uploads, RGB conversion and malformed data rejection.
for mode in ['L','RGBA']:
    buffer=BytesIO();Image.new(mode,(70,30)).save(buffer,format='PNG');buffer.seek(0)
    assert tuple(load_image(buffer).shape)==(1,3,256,256)
for payload in [b'bad image',b'x'*(5*1024*1024+1)]:
    try:load_image(BytesIO(payload))
    except ValueError:pass
    else:raise AssertionError('Invalid upload accepted')
try:generate_image(next(iter(models.values())),torch.zeros(1,256,256,3))
except ValueError:pass
else:raise AssertionError('Invalid tensor shape accepted')
print('All four original images match independent torchvision preprocessing and direct generator results.')
print('Grayscale/RGBA uploads, corrupt/oversized uploads and invalid tensor-shape regressions passed.')
workspace.cleanup()
print('Owned temporary workspace cleaned up.')

## Original source code

These blocks are preserved verbatim for comparison, not executed.

### Original block 1

```text
python -m venv gan_application
```

### Original block 2

```text
.\gan_application\scripts\activate
```

### Original block 3

```text
pip install -r requirements.txt
```

### Original block 4

```text
import streamlit as st
from PIL import Image
from tensorflow.keras.models import load_model
from keras_contrib.layers.normalization.instancenormalization import InstanceNormalization 
import numpy as np
from numpy import  vstack
```

### Original block 5

```text
#loading pretrained GAN models 
cust = {'InstanceNormalization':InstanceNormalization}
model_horse2zebra = load_model('./g_model_AtoB_023740.h5',cust)
model_zebra2horse = load_model('./g_model_BtoA_023740.h5',cust)
```

### Original block 6

```text
def load_image(image_path):
    """
    Function to load image given image path
    :param image_path: image path 
    :return: loaded image
    """
    image = Image.open(image_path)
    newsize = (256, 256)
    image = image.resize(newsize)
    image = np.array(image)
    image = image[np.newaxis, ...]  # convert the array into 3D array
    return image
```

### Original block 7

```text
def generate_image(model, image):
    """
    Function to load image given image path
    :param model: The GAN model 
    :param image: The image to apply the GAN model on
    :return: generated images
    """
    generated_image = model.predict(image)
    images = vstack(generated_image)
    images = (images+1)/2.0
    return images
```

### Original block 8

```text
# Building the streamlit application 

st.title("Horse Zebra GAN Web APP") # Set the title  
st.image('./horses_zebras.png') # set the featured image of the web application
```

### Original block 9

```text
pick = st.selectbox("Please select a GAN model to use:", ["Horse 2 Zebra GAN", "Zebra 2 Horse GAN"])
```

### Original block 10

```text
if pick == "Horse 2 Zebra GAN": # if the selected case is to convert from horse to zebra
    st.write("This is a GAN model for Generating Zebra images from Horses")
    st.write("Try out the GAN model with a default images of a horse or simply upload an image")

    if st.button("Try with Default Image"): # if the user selected to use the default image
        image = load_image('./horse.jpg')
        st.subheader("Horse Image")
        st.image(image)
        st.subheader("Generated Zebra Image")
        st.image(generate_image(model_horse2zebra, image))

    st.subheader("Upload an image file of a horse to convert it to a Zebra") # if the user selected to upload an image
    uploaded_file = st.file_uploader("Upload JPG image file of a horse only", type=["jpg","jpeg"]) # only "jpg" and "jpeg" is allaowed.

    if uploaded_file:
        image = load_image(uploaded_file)
        st.image(generate_image(model_horse2zebra, image)) # generate the new image 

else: # if the second case was selected 
    st.write("This is a GAN model for Generating Horse images from Zebras")
    st.write("Try out the GAN model with a default images of a zebra or simply upload an image")

    if st.button("Try with Default Image"):
        image = load_image('./zebra.jpg')
        st.subheader("Horse Image")
        st.image(image)
        st.subheader("Generated Horse Image")
        st.image(generate_image(model_zebra2horse, image))

    st.subheader("Upload an image file of a zebra to convert it to a horse")
    uploaded_file = st.file_uploader("Upload JPG image file of a zebra only", type=["jpg","jpeg"])

    if uploaded_file:
        image = load_image(uploaded_file)
        st.image(generate_image(model_zebra2horse, image))
```

### Original block 11

```text
streamlit run app.py
```